# Projeto Aplicado III - Entrega 3

## Sistema inteligente de recomendação de receitas para redução do desperdício alimentar

Este notebook apresenta o passo a passo da terceira entrega do projeto.

A ideia aqui é manter o código simples e fácil de explicar, sem criar uma estrutura robusta de software.  
O foco é mostrar:

1. análise dos resultados preliminares da etapa anterior;
2. ajustes simples no pipeline;
3. nova avaliação do modelo;
4. organização da metodologia usada no projeto.

## 1. Bibliotecas utilizadas

Nesta etapa foram usadas bibliotecas comuns de análise de dados e aprendizado de máquina:

- `pandas`: leitura e manipulação das bases;
- `numpy`: cálculos numéricos;
- `scikit-learn`: vetorização TF-IDF e métricas;
- `ast`: conversão de listas gravadas como texto;
- `math`: cálculo da raiz quadrada para o RMSE.

In [1]:
import ast
import math
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_absolute_error, mean_squared_error

ModuleNotFoundError: No module named 'pandas'

## 2. Caminho dos arquivos

No GitHub, a recomendação é deixar os arquivos dentro da pasta `data`.

No ambiente do ChatGPT, os arquivos estão em `/mnt/data`.  
Por isso, o código abaixo tenta usar primeiro a pasta `data`. Caso ela não exista, usa `/mnt/data`.

In [ ]:
pasta_dados = Path('data')

if not pasta_dados.exists():
    pasta_dados = Path('/mnt/data')

arquivo_treino = pasta_dados / 'interactions_train.csv'
arquivo_validacao = pasta_dados / 'interactions_validation.csv'
arquivo_teste = pasta_dados / 'interactions_test.csv'
arquivo_receitas = pasta_dados / 'PP_recipes.csv'

print('Pasta usada:', pasta_dados)
print('Arquivo de treino existe?', arquivo_treino.exists())
print('Arquivo de validação existe?', arquivo_validacao.exists())
print('Arquivo de teste existe?', arquivo_teste.exists())
print('Arquivo de receitas existe?', arquivo_receitas.exists())

## 3. Carregamento dos dados

Nesta etapa são carregados os arquivos já separados em treino, validação e teste.

A base `PP_recipes.csv` contém informações processadas das receitas.  
As bases de interações contêm usuários, receitas e avaliações.

In [ ]:
treino = pd.read_csv(arquivo_treino, usecols=['user_id', 'recipe_id', 'rating', 'i'])
validacao = pd.read_csv(arquivo_validacao, usecols=['user_id', 'recipe_id', 'rating', 'i'])
teste = pd.read_csv(arquivo_teste, usecols=['user_id', 'recipe_id', 'rating', 'i'])

receitas = pd.read_csv(arquivo_receitas, usecols=['i', 'name_tokens', 'ingredient_ids', 'calorie_level'])

print('Treino:', treino.shape)
print('Validação:', validacao.shape)
print('Teste:', teste.shape)
print('Receitas:', receitas.shape)

## 4. Análise exploratória simples

Antes de treinar o modelo, é importante entender a quantidade de usuários, receitas e avaliações.

Também é importante observar a distribuição das notas, pois sistemas de recomendação com avaliações muito concentradas em notas altas podem gerar desafios na avaliação.

In [ ]:
print('Usuários no treino:', treino['user_id'].nunique())
print('Receitas no treino:', treino['recipe_id'].nunique())
print('Interações no treino:', len(treino))

print('\nDistribuição das notas no treino:')
display(treino['rating'].value_counts().sort_index())

print('\nMédia das notas:', round(treino['rating'].mean(), 4))
print('Mediana das notas:', treino['rating'].median())

## 5. Esparsidade da matriz usuário-receita

A esparsidade indica o quanto a matriz usuário-receita é vazia.

Em sistemas de recomendação, é comum que um usuário avalie poucas receitas em relação ao total de receitas disponíveis.  
Isso dificulta a recomendação, principalmente quando se usa apenas filtragem colaborativa.

In [ ]:
qtd_usuarios = treino['user_id'].nunique()
qtd_receitas = treino['recipe_id'].nunique()
qtd_interacoes = len(treino)

total_possivel = qtd_usuarios * qtd_receitas
densidade = qtd_interacoes / total_possivel
esparsidade = 1 - densidade

print('Densidade:', round(densidade, 6))
print('Esparsidade:', round(esparsidade, 6))

## 6. Resultado preliminar da etapa anterior

Na etapa anterior, a prova de conceito mostrou que a recomendação por similaridade de conteúdo funcionava para encontrar receitas parecidas.

Porém, a avaliação Top-K inicial ficou muito baixa, chegando a Precision@10 e Recall@10 iguais a 0 em alguns testes.  
Isso aconteceu principalmente por causa da alta esparsidade da base: existem muitos usuários e muitas receitas, mas poucas avaliações por usuário.

Por isso, nesta terceira etapa foram feitos ajustes simples:

- usar corretamente os arquivos de treino, validação e teste;
- criar uma linha de base por média do usuário e média da receita;
- montar uma recomendação híbrida simples;
- combinar conteúdo dos ingredientes com popularidade da receita;
- avaliar novamente com RMSE, MAE, Precision@10, Recall@10 e HitRate@10.

## 7. Linha de base para previsão de notas

A primeira avaliação usa uma ideia simples:

- calcular a média de nota de cada usuário;
- calcular a média de nota de cada receita;
- combinar as duas médias.

Essa abordagem não é sofisticada, mas serve como comparação inicial.

In [ ]:
media_geral = treino['rating'].mean()
media_por_usuario = treino.groupby('user_id')['rating'].mean()
media_por_receita = treino.groupby('recipe_id')['rating'].mean()

print('Média geral das notas:', round(media_geral, 4))

In [ ]:
def avaliar_previsao_notas(dados_avaliacao):
    previsao_usuario = dados_avaliacao['user_id'].map(media_por_usuario).fillna(media_geral)
    previsao_receita = dados_avaliacao['recipe_id'].map(media_por_receita).fillna(media_geral)

    previsao_final = (previsao_usuario + previsao_receita) / 2

    rmse = math.sqrt(mean_squared_error(dados_avaliacao['rating'], previsao_final))
    mae = mean_absolute_error(dados_avaliacao['rating'], previsao_final)

    return rmse, mae

In [ ]:
rmse_validacao, mae_validacao = avaliar_previsao_notas(validacao)
rmse_teste, mae_teste = avaliar_previsao_notas(teste)

print('Validação - RMSE:', round(rmse_validacao, 4))
print('Validação - MAE:', round(mae_validacao, 4))

print('\nTeste - RMSE:', round(rmse_teste, 4))
print('Teste - MAE:', round(mae_teste, 4))

## 8. Preparação das receitas para recomendação por conteúdo

Agora será criada uma representação textual simples para cada receita.

Neste projeto, foram usados:

- ingredientes da receita;
- nível calórico.

A coluna `ingredient_ids` vem como texto representando uma lista.  
Por isso, foi criada uma função simples para transformar essa lista em texto.

In [ ]:
def transformar_lista_em_texto(valor):
    try:
        lista = ast.literal_eval(valor)
    except:
        return ''

    if isinstance(lista, list):
        textos = []
        for item in lista:
            textos.append('ingrediente_' + str(item))
        return ' '.join(textos)

    return ''

In [ ]:
receitas_modelo = receitas.copy()
receitas_modelo = receitas_modelo.drop_duplicates('i')
receitas_modelo['texto_receita'] = receitas_modelo['ingredient_ids'].apply(transformar_lista_em_texto)

receitas_modelo['texto_receita'] = (
    receitas_modelo['texto_receita'] 
    + ' nivel_caloria_' 
    + receitas_modelo['calorie_level'].astype(str)
)

display(receitas_modelo[['i', 'ingredient_ids', 'calorie_level', 'texto_receita']].head())

## 9. Seleção de receitas candidatas

Para deixar o notebook mais leve, não vamos comparar todas as receitas com todas as receitas.

A estratégia simples é selecionar:

- receitas que aparecem na validação;
- receitas que aparecem no teste;
- receitas curtidas pelos usuários no treino;
- receitas populares no treino.

Isso reduz o tamanho da matriz e permite executar o notebook em computador comum.

In [ ]:
qtd_receitas_populares = 5000

usuarios_avaliacao = set(validacao['user_id']) | set(teste['user_id'])

curtidas_treino = treino[
    (treino['rating'] >= 4) &
    (treino['user_id'].isin(usuarios_avaliacao))
]

estatisticas_receita = treino.groupby('i')['rating'].agg(['mean', 'count'])
estatisticas_receita['popularidade'] = estatisticas_receita['mean'] * np.log1p(estatisticas_receita['count'])

receitas_populares = set(
    estatisticas_receita
    .sort_values('popularidade', ascending=False)
    .head(qtd_receitas_populares)
    .index
)

receitas_candidatas = set(validacao['i']) | set(teste['i']) | set(curtidas_treino['i']) | receitas_populares

receitas_modelo = receitas_modelo[receitas_modelo['i'].isin(receitas_candidatas)]
receitas_modelo = receitas_modelo.reset_index(drop=True)

print('Receitas candidatas:', len(receitas_modelo))

## 10. Vetorização TF-IDF

O TF-IDF transforma o texto da receita em números.

Assim, receitas que compartilham ingredientes parecidos tendem a ter vetores mais parecidos.

In [ ]:
vetorizador = TfidfVectorizer(min_df=1)
matriz_receitas = vetorizador.fit_transform(receitas_modelo['texto_receita'])

print('Formato da matriz:', matriz_receitas.shape)

## 11. Dicionários auxiliares

Nesta etapa são criados dicionários simples para facilitar a recomendação:

- posição de cada receita na matriz;
- popularidade normalizada;
- receitas curtidas por usuário;
- receitas já vistas por usuário.

In [ ]:
posicao_receita = {}

for posicao, codigo in enumerate(receitas_modelo['i']):
    posicao_receita[int(codigo)] = posicao

popularidade = np.zeros(len(receitas_modelo))

for codigo_receita, valor in estatisticas_receita['popularidade'].items():
    if codigo_receita in posicao_receita:
        popularidade[posicao_receita[codigo_receita]] = valor

if popularidade.max() > 0:
    popularidade = popularidade / popularidade.max()

curtidas_por_usuario = curtidas_treino.groupby('user_id')['i'].apply(list).to_dict()
vistas_por_usuario = treino[treino['user_id'].isin(usuarios_avaliacao)].groupby('user_id')['i'].apply(set).to_dict()

print('Usuários com receitas curtidas:', len(curtidas_por_usuario))
print('Usuários com receitas vistas:', len(vistas_por_usuario))

## 12. Recomendador híbrido simples

A recomendação híbrida combina duas pontuações:

- **conteúdo**: similaridade entre os ingredientes das receitas que o usuário gostou e as receitas candidatas;
- **popularidade**: receitas bem avaliadas e com mais avaliações recebem maior peso.

A fórmula usada foi:

`pontuação final = 65% conteúdo + 35% popularidade`

Esse ajuste foi escolhido porque o objetivo do projeto é recomendar receitas compatíveis com ingredientes e preferências, mas sem ignorar o comportamento geral dos usuários.

In [ ]:
def pontuar_receitas_usuario(usuario, candidatos):
    candidatos_validos = []

    for receita in candidatos:
        ja_vista = receita in vistas_por_usuario.get(usuario, set())
        existe_no_modelo = receita in posicao_receita

        if existe_no_modelo and not ja_vista:
            candidatos_validos.append(receita)

    if len(candidatos_validos) == 0:
        return []

    posicoes_candidatas = []
    for receita in candidatos_validos:
        posicoes_candidatas.append(posicao_receita[receita])

    receitas_curtidas = []
    for receita in curtidas_por_usuario.get(usuario, [])[-30:]:
        if receita in posicao_receita:
            receitas_curtidas.append(receita)

    if len(receitas_curtidas) > 0:
        posicoes_curtidas = []
        for receita in receitas_curtidas:
            posicoes_curtidas.append(posicao_receita[receita])

        perfil_usuario = matriz_receitas[posicoes_curtidas].mean(axis=0)
        pontuacao_conteudo = np.asarray(perfil_usuario @ matriz_receitas[posicoes_candidatas].T).ravel()

        if pontuacao_conteudo.max() > 0:
            pontuacao_conteudo = pontuacao_conteudo / pontuacao_conteudo.max()

        pontuacao_final = 0.65 * pontuacao_conteudo + 0.35 * popularidade[posicoes_candidatas]
    else:
        pontuacao_final = popularidade[posicoes_candidatas]

    ordem = np.argsort(-pontuacao_final)

    receitas_ordenadas = []
    for posicao in ordem:
        receitas_ordenadas.append(candidatos_validos[posicao])

    return receitas_ordenadas

## 13. Avaliação Top-K

Nesta avaliação, consideramos como relevante uma receita com nota maior ou igual a 4.

Para cada usuário, o modelo recebe um conjunto de receitas candidatas e tenta colocar as receitas relevantes entre as primeiras recomendações.

As métricas usadas foram:

- **Precision@10**: proporção de recomendações corretas entre as 10 primeiras;
- **Recall@10**: proporção de receitas relevantes recuperadas entre as 10 primeiras;
- **HitRate@10**: proporção de usuários para os quais o sistema acertou pelo menos uma receita.

In [ ]:
def avaliar_top_k(dados_avaliacao, qtd_usuarios_avaliacao=1000, qtd_negativos_por_usuario=100, k=10):
    gerador = np.random.default_rng(42)

    receitas_relevantes = dados_avaliacao[dados_avaliacao['rating'] >= 4]
    relevantes_por_usuario = receitas_relevantes.groupby('user_id')['i'].apply(set).to_dict()

    usuarios = []
    for usuario in relevantes_por_usuario.keys():
        if usuario in curtidas_por_usuario:
            usuarios.append(usuario)

    usuarios = usuarios[:qtd_usuarios_avaliacao]

    lista_receitas_populares = list(
        estatisticas_receita
        .sort_values('popularidade', ascending=False)
        .head(qtd_receitas_populares)
        .index
    )

    precisoes = []
    revocacoes = []
    acertos_usuarios = 0

    for usuario in usuarios:
        relevantes = set()

        for receita in relevantes_por_usuario[usuario]:
            if receita in posicao_receita:
                relevantes.add(receita)

        if len(relevantes) == 0:
            continue

        negativos = []

        for receita in lista_receitas_populares:
            nao_relevante = receita not in relevantes
            nao_vista = receita not in vistas_por_usuario.get(usuario, set())
            existe_no_modelo = receita in posicao_receita

            if nao_relevante and nao_vista and existe_no_modelo:
                negativos.append(receita)

        if len(negativos) > qtd_negativos_por_usuario:
            negativos = list(gerador.choice(negativos, size=qtd_negativos_por_usuario, replace=False))

        candidatos = list(relevantes) + negativos

        recomendadas = pontuar_receitas_usuario(usuario, candidatos)[:k]

        acertos = len(set(recomendadas) & relevantes)

        precisoes.append(acertos / k)
        revocacoes.append(acertos / len(relevantes))

        if acertos > 0:
            acertos_usuarios += 1

    if len(precisoes) == 0:
        return 0, 0, 0, 0

    precision_k = np.mean(precisoes)
    recall_k = np.mean(revocacoes)
    hit_rate_k = acertos_usuarios / len(precisoes)

    return len(precisoes), precision_k, recall_k, hit_rate_k

In [ ]:
n_validacao, precision_validacao, recall_validacao, hit_validacao = avaliar_top_k(validacao)

print('Validação')
print('Usuários avaliados:', n_validacao)
print('Precision@10:', round(precision_validacao, 4))
print('Recall@10:', round(recall_validacao, 4))
print('HitRate@10:', round(hit_validacao, 4))

In [ ]:
n_teste, precision_teste, recall_teste, hit_teste = avaliar_top_k(teste)

print('Teste')
print('Usuários avaliados:', n_teste)
print('Precision@10:', round(precision_teste, 4))
print('Recall@10:', round(recall_teste, 4))
print('HitRate@10:', round(hit_teste, 4))

## 14. Comparação dos resultados

A tabela abaixo reúne os principais resultados da reavaliação.

O RMSE e o MAE avaliam a previsão de notas.  
Precision@10, Recall@10 e HitRate@10 avaliam a lista de recomendações.

In [ ]:
resultados = pd.DataFrame({
    'base': ['validacao', 'teste'],
    'rmse': [rmse_validacao, rmse_teste],
    'mae': [mae_validacao, mae_teste],
    'precision_10': [precision_validacao, precision_teste],
    'recall_10': [recall_validacao, recall_teste],
    'hit_rate_10': [hit_validacao, hit_teste],
    'usuarios_avaliados_top_k': [n_validacao, n_teste]
})

display(resultados)

## 15. Exemplo de recomendação para um usuário

Abaixo é mostrado um exemplo simples de recomendação.

O código escolhe um usuário da validação e recomenda receitas entre as candidatas avaliadas.

In [ ]:
usuario_exemplo = validacao['user_id'].iloc[0]

candidatos_exemplo = list(validacao['i'].head(500))
recomendadas_exemplo = pontuar_receitas_usuario(usuario_exemplo, candidatos_exemplo)[:10]

print('Usuário exemplo:', usuario_exemplo)
print('Receitas recomendadas:', recomendadas_exemplo)

In [ ]:
nomes_receitas = receitas_modelo[['i', 'name_tokens']].copy()
recomendacoes_exemplo = pd.DataFrame({'i': recomendadas_exemplo})
recomendacoes_exemplo = recomendacoes_exemplo.merge(nomes_receitas, on='i', how='left')

display(recomendacoes_exemplo)

## 16. Conclusão da Etapa 3

A terceira etapa mostrou que a prova de conceito da etapa anterior precisava de ajustes.

O principal problema encontrado foi a alta esparsidade da matriz usuário-receita.  
Por isso, a avaliação Top-K é difícil, pois há muitas receitas possíveis e poucas avaliações por usuário.

Os ajustes feitos foram simples, mas importantes:

- uso correto das bases de treino, validação e teste;
- criação de uma linha de base por média do usuário e média da receita;
- construção de um modelo híbrido simples;
- combinação entre conteúdo dos ingredientes e popularidade;
- reavaliação com métricas de erro e métricas de recomendação.

Mesmo com melhora em relação ao caso em que Precision@10 e Recall@10 ficaram zerados, os resultados Top-K ainda são baixos.  
Isso indica que, na próxima etapa, o projeto deve discutir limitações, resultados finais e possíveis melhorias, como testar outras formas de recomendação híbrida, ajustar pesos, melhorar o tratamento de usuários com poucas interações e incorporar a regra de ingredientes próximos da validade.